In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score # métrica de evaluación
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df_nasa = pd.read_csv('https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/asteroids_nasa.csv')
df_nasa.head(5)

Salteo el análisis ya que es un dataset que conocemos
# Preprocesamiento de datos

In [ ]:
X = df_nasa.drop(columns="Hazardous")
y = df_nasa["Hazardous"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

columnas = [ 'Absolute Magnitude', 'Est Dia in KM(min)',
       'Est Dia in KM(max)', 'Est Dia in M(min)', 'Est Dia in M(max)',
       'Est Dia in Miles(min)', 'Est Dia in Miles(max)',
       'Est Dia in Feet(min)', 'Est Dia in Feet(max)', 'Relative Velocity km per sec',
       'Relative Velocity km per hr', 'Miles per hour',
       'Miss Dist.(Astronomical)', 'Miss Dist.(lunar)',
       'Miss Dist.(kilometers)', 'Miss Dist.(miles)',
       'Orbit ID', 'Orbit Uncertainity','Minimum Orbit Intersection', 'Jupiter Tisserand Invariant',
       'Epoch Osculation', 'Eccentricity', 'Semi Major Axis', 'Inclination',
       'Asc Node Longitude', 'Orbital Period', 'Perihelion Distance',
       'Perihelion Arg', 'Aphelion Dist', 'Perihelion Time', 'Mean Anomaly','Mean Motion']


In [ ]:
pl = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", columnas)], remainder="drop")),
    ('classifier', DecisionTreeClassifier(max_depth=2, random_state=1))
])
pl.fit(X_train, y_train)

In [ ]:
y_pred_tc = pl.predict(X_test)

#Exactitud del modelo
print('Exactitud (accuracy) del modelo: {:.2f} %'.format(accuracy_score(y_test, y_pred_tc)*100))
print("-"*100)

# Reporte del clasificador
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred_tc))

# Comparamos por curva ROC los modelos

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

pl_knn = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", columnas)], remainder="drop")),
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('classifier', KNeighborsClassifier(n_neighbors=11))
])
pl_knn.fit(X_train, y_train)
y_pred_knn = pl_knn.predict(X_test)

pl_NBg = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", columnas)], remainder="drop")),
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('classifier', GaussianNB())
])
pl_NBg.fit(X_train, y_train)
y_pred_gauss = pl_NBg.predict(X_test)

pl_NGm = Pipeline([
    ("selector", ColumnTransformer([("selector", "passthrough", columnas)], remainder="drop")),
    ('classifier', MultinomialNB())
])
pl_NGm.fit(X_train, y_train)
y_pred_nb = pl_NGm.predict(X_test)

In [ ]:
def graficarCurvaRoc( y_pred, model ):
  fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred)
  auc = metrics.roc_auc_score(y_test, y_pred)
  # Graficamos
  plt.plot(fpr,tpr,label= model +" AUC="+str(round(auc,4))) #,label= "AUC="+str(auc))
  plt.legend(loc=4, fontsize=12)
  return auc

# Inicializamos los labels del gráfico
plt.figure(figsize=(20, 10))
plt.xlabel('% Not Hazardous', fontsize=14)
plt.ylabel('% Hazardous', fontsize=14)

# Graficamos la recta del azar
it = [i/100 for i in range(100)]
plt.plot(it,it,label="AZAR AUC=0.5",color="black")

modelos = {'bayesMulti':y_pred_nb, 'bayesGauss':y_pred_gauss,
             'arbol':y_pred_tc,'knn':y_pred_knn}
for pred in modelos:
    auc = graficarCurvaRoc( modelos[pred] , pred )

# Agregamos el titulo y configuro el tamaño de letra
plt.title("Curva ROC", fontsize=14)
plt.tick_params(labelsize=12);
plt.show()

# Mejoramos el modelo de Arbol de Decision con GridSearch

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        "classifier__max_depth": [2, 5, 7, 10, None],
        "classifier__min_samples_leaf": [2, 10, 50, 75, 100, 200],
        "classifier__criterion": ['gini', 'entropy', 'log_loss'],
        "classifier": [DecisionTreeClassifier()]
    }]

grid_search = GridSearchCV(pl, param_grid, cv=10, verbose=1,n_jobs=-1, scoring='roc_auc')
grid_search.fit(X_train, y_train)

In [ ]:
y_pred_gsCV = grid_search.best_estimator_.predict(X_test)
print(grid_search.best_params_)

In [ ]:
graficarCurvaRoc(y_pred_gsCV, "GridSearch Tree")

# Ejercicio
- Usar el step de "selector" del pipeline para remover las columnas redundantes
- Crear un pipeline para el clasificador randomforest
- Usar GridSearchCV para encontrar el valor optimo de k en KNN